In [ ]:
#titanic survival prediction using KNN
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
train_df=pd.read_csv("/content/train.csv")

In [ ]:
train_df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [ ]:
train_df.duplicated().sum()

np.int64(0)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [ ]:
#droping unimportant columns before model fiting
train_df=train_df.drop(columns=['PassengerId','Ticket','Cabin'])

In [ ]:
train_df.head(2)#name aslo not much important for survival prediction.let's drop name column also

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C


In [ ]:
train_df=train_df.drop(columns=['Name'])

In [ ]:
train_df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [ ]:
train_df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [ ]:
#lets fill age column with median and Embarked with mode
train_df['Age']=train_df['Age'].fillna(train_df['Age'].median())
train_df['Embarked']=train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

In [ ]:
train_df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [ ]:
#now encode the categorical variables and scale the numerical variables
cat_data=train_df.select_dtypes(exclude=np.number)
num_data=train_df.select_dtypes(include=np.number)

In [ ]:
cat_data.head(2)

,Sex,Embarked
0,male,S
1,female,C


In [ ]:
for col in cat_data:
 print(cat_data[col].value_counts())

Sex
male      577
female    314
Name: count, dtype: int64
Embarked
S    646
C    168
Q     77
Name: count, dtype: int64


In [ ]:
#we will use dummy encoding here
cat_data=pd.get_dummies(cat_data,dtype=int,drop_first=True)

In [ ]:
#for scaling we will do after splitting to avoide data leakege
final_df=pd.concat([cat_data,num_data],axis=1)

In [ ]:
final_df.head(2)

,Sex_male,Embarked_Q,Embarked_S,Survived,Pclass,Age,SibSp,Parch,Fare
0,1,0,1,0,3,22.0,1,0,7.2500
1,0,0,0,1,1,38.0,1,0,71.2833


In [ ]:
x=final_df.drop(columns=['Survived'])
y=final_df['Survived']

In [ ]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.15,random_state=42)

In [ ]:
print(xtrain.shape)
print(xtest.shape)
print(ytrain.shape)
print(ytest.shape)

(757, 8)
(134, 8)
(757,)
(134,)


In [ ]:
#now we will performe scaling on train data and apply same scalling on test data
xtrain.head()

,Sex_male,Embarked_Q,Embarked_S,Pclass,Age,SibSp,Parch,Fare
599,1,0,0,1,49.0,1,0,56.9292
830,0,0,0,3,15.0,1,0,14.4542
306,0,0,0,1,28.0,0,0,110.8833
231,1,0,1,3,29.0,0,0,7.7750
845,1,0,1,3,42.0,0,0,7.5500


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

xtrain[['Age','SibSp','Parch','Fare']]=scaler.fit_transform(xtrain[['Age','SibSp','Parch','Fare']])
xtest[['Age','SibSp','Parch','Fare']]=scaler.transform(xtest[['Age','SibSp','Parch','Fare']])


In [ ]:
xtrain.head()

,Sex_male,Embarked_Q,Embarked_S,Pclass,Age,SibSp,Parch,Fare
599,1,0,0,1,1.528337,0.400772,-0.468859,0.486264
830,0,0,0,3,-1.100894,0.400772,-0.468859,-0.348553
306,0,0,0,1,-0.095599,-0.471022,-0.468859,1.546694
231,1,0,1,3,-0.018269,-0.471022,-0.468859,-0.479828
845,1,0,1,3,0.987025,-0.471022,-0.468859,-0.484251


In [ ]:
#KNN model bulding
from sklearn.neighbors import KNeighborsClassifier
KNclassi=KNeighborsClassifier(n_neighbors=5,weights='distance',p=1)
KNclassi.fit(xtrain,ytrain)
model_pred=KNclassi.predict(xtest)

In [ ]:
#model evalution
from sklearn.metrics import classification_report,roc_auc_score
print(classification_report(ytest,model_pred))
print(roc_auc_score(ytest,model_pred))

              precision    recall  f1-score   support

           0       0.83      0.83      0.83        78
           1       0.77      0.77      0.77        56

    accuracy                           0.81       134
   macro avg       0.80      0.80      0.80       134
weighted avg       0.81      0.81      0.81       134

0.8005952380952381


In [ ]:
#importing the test data and predictions for testing data
test_df=pd.read_csv("/content/test.csv")
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
test_df.duplicated().sum()

np.int64(0)

In [ ]:
#remoing the columns which we removed in model training
test_df=test_df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [ ]:
test_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,34.5,0,0,7.8292,Q
1,3,female,47.0,1,0,7.0000,S
2,2,male,62.0,0,0,9.6875,Q
3,3,male,27.0,0,0,8.6625,S
4,3,female,22.0,1,1,12.2875,S


In [ ]:
test_df.isnull().sum()

,0
Pclass,0
Sex,0
Age,86
SibSp,0
Parch,0
Fare,1
Embarked,0


In [ ]:
#filling the null values
test_df['Age']=test_df['Age'].fillna(test_df['Age'].median())
test_df['Fare']=test_df['Fare'].fillna(test_df['Fare'].median())

In [ ]:
test_df.isnull().sum()

,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [ ]:
#encodding and scalling
cate_data=test_df.select_dtypes(exclude=np.number)
num_data=test_df.select_dtypes(include=np.number)

In [ ]:
cate_data.head(2)

,Sex,Embarked
0,male,Q
1,female,S


In [ ]:
encode_data=pd.get_dummies(cate_data,dtype=int,drop_first=True)

In [ ]:
encode_data.head(2)

,Sex_male,Embarked_Q,Embarked_S
0,1,1,0
1,0,0,1


In [ ]:
num_data.head(2)

,Pclass,Age,SibSp,Parch,Fare
0,3,34.5,0,0,7.8292
1,3,47.0,1,0,7.0000


In [ ]:
num_data[['Age','SibSp','Parch','Fare']]=scaler.transform(num_data[['Age','SibSp','Parch','Fare']])

In [ ]:
num_data.head(2)

,Pclass,Age,SibSp,Parch,Fare
0,3,0.407048,-0.471022,-0.468859,-0.478763
1,3,1.373677,0.400772,-0.468859,-0.495060


In [ ]:
final_test_df=pd.concat([encode_data,num_data],axis=1)

In [ ]:
KNNmodel_pred=KNclassi.predict(final_test_df)

In [ ]:
len(KNNmodel_pred)

418

In [ ]:
test_df=pd.read_csv("/content/test.csv")


In [ ]:
predictions_KNN=pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':KNNmodel_pred})

In [ ]:
predictions_KNN.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0


In [ ]:
predictions_KNN.to_csv('predictions_KNN.csv',index=False)

In [ ]:
from sklearn.model_selection import GridSearchCV
parameters={'n_neighbors':[3,4,5,6,7],'p':[1,2,3],'weights':['uniform','distance']}
knn_classifier=KNeighborsClassifier()
clf=GridSearchCV(knn_classifier,parameters,cv=3,n_jobs=-1)
clf.fit(xtrain,ytrain)

GridSearchCV(cv=3, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'n_neighbors': [3, 4, 5, 6, 7], 'p': [1, 2, 3],
                         'weights': ['uniform', 'distance']})

In [ ]:
clf.best_params_

{'n_neighbors': 6, 'p': 1, 'weights': 'uniform'}

In [ ]:
y_pred = clf.predict(xtest)

y_prob = clf.predict_proba(xtest)[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))
print("ROC-AUC:", roc_auc_score(ytest, y_prob))

Accuracy: 0.8208955223880597
              precision    recall  f1-score   support

           0       0.81      0.90      0.85        78
           1       0.83      0.71      0.77        56

    accuracy                           0.82       134
   macro avg       0.82      0.81      0.81       134
weighted avg       0.82      0.82      0.82       134

ROC-AUC: 0.8712225274725275


In [ ]:
KNNgridserchCV_pred=clf.predict(final_test_df)

In [ ]:
KNNgridserchCV_pred

array([0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [ ]:
prediction_KNNGSCV=pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':KNNgridserchCV_pred})

In [ ]:
prediction_KNNGSCV.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0


In [ ]:
prediction_KNNGSCV.to_csv('prediction_KNNGSCV.csv',index=False)